<a href="https://colab.research.google.com/github/Andrea180904/Big_Data/blob/main/lab_sparksql_and_dataframes_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [5]:
# Cache the dataframe since we will be running many aggregations over it,
# and take a quick look at the schema and row count
df_trips.cache()
df_trips.printSchema()
print(f"Number of rows: {df_trips.count():,}")

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)

Number of rows: 7,696,617


## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

#### Add a unique key to identify each record

In [6]:
from pyspark.sql.functions import monotonically_increasing_id

# monotonically_increasing_id() guarantees a unique (though not necessarily
# consecutive) id per row without requiring a shuffle, which is ideal for a
# dataset this size
df_trips = df_trips.withColumn("trip_id", monotonically_increasing_id())
df_trips.select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime").show(5)

+-------+--------------------+---------------------+
|trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------+--------------------+---------------------+
|      0| 2019-01-01 00:46:40|  2019-01-01 00:53:20|
|      1| 2019-01-01 00:59:47|  2019-01-01 01:18:59|
|      2| 2018-12-21 13:48:30|  2018-12-21 13:52:40|
|      3| 2018-11-28 15:52:25|  2018-11-28 15:55:45|
|      4| 2018-11-28 15:56:57|  2018-11-28 15:58:33|
+-------+--------------------+---------------------+
only showing top 5 rows


#### Which trip has the highest passenger count

In [7]:
from pyspark.sql.functions import desc

df_trips.orderBy(desc("passenger_count")).select(
    "trip_id", "passenger_count", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "trip_distance", "total_amount"
).show(5)

+-------+---------------+--------------------+---------------------+-------------+------------+
|trip_id|passenger_count|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|total_amount|
+-------+---------------+--------------------+---------------------+-------------+------------+
|2012098|            9.0| 2019-01-10 00:43:10|  2019-01-10 00:43:14|          0.0|        11.3|
|7286683|            9.0| 2019-01-30 18:34:12|  2019-01-30 18:34:16|          0.0|        10.3|
|2883995|            9.0| 2019-01-13 04:13:24|  2019-01-13 04:14:34|          0.0|       12.25|
|1296287|            9.0| 2019-01-07 03:19:36|  2019-01-07 03:20:01|          0.0|         9.3|
|4534707|            9.0| 2019-01-19 16:45:25|  2019-01-19 16:45:27|          0.0|      110.76|
+-------+---------------+--------------------+---------------------+-------------+------------+
only showing top 5 rows


#### What is the average passenger count

In [8]:
from pyspark.sql.functions import avg, round as spark_round

df_trips.select(spark_round(avg("passenger_count"), 2).alias("avg_passenger_count")).show()

+-------------------+
|avg_passenger_count|
+-------------------+
|               1.57|
+-------------------+



#### Shortest / longest trip by distance

In [9]:
from pyspark.sql.functions import col

# A trip_distance of 0 is not a meaningful "shortest" trip, so we filter those
# out first and look separately at the true minimum vs. zero-distance records
print("Longest trip by distance:")
df_trips.orderBy(desc("trip_distance")).select(
    "trip_id", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime"
).show(5)

print("Shortest trip by distance (excluding 0-mile trips):")
df_trips.filter(col("trip_distance") > 0).orderBy("trip_distance").select(
    "trip_id", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime"
).show(5)

zero_distance_count = df_trips.filter(col("trip_distance") == 0).count()
print(f"Trips with a recorded distance of exactly 0: {zero_distance_count:,}")

Longest trip by distance:
+-------+-------------+--------------------+---------------------+
|trip_id|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------+-------------+--------------------+---------------------+
|6074091|        831.8| 2019-01-25 21:56:39|  2019-01-25 22:06:08|
|4286633|        700.7| 2019-01-18 16:32:24|  2019-01-18 16:39:20|
|6770985|       214.01| 2019-01-28 17:24:11|  2019-01-29 04:37:16|
|4707534|       211.36| 2019-01-20 12:22:24|  2019-01-20 17:05:36|
|4881785|       201.27| 2019-01-21 10:13:05|  2019-01-21 13:28:01|
+-------+-------------+--------------------+---------------------+
only showing top 5 rows
Shortest trip by distance (excluding 0-mile trips):
+-------+-------------+--------------------+---------------------+
|trip_id|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------+-------------+--------------------+---------------------+
|  18828|         0.01| 2019-01-01 01:03:20|  2019-01-01 01:04:01|
|  22467|         0.01| 201

#### Shortest / longest trip by time

In [10]:
from pyspark.sql.functions import unix_timestamp

df_trips = df_trips.withColumn(
    "trip_duration_min",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)

print("Longest trip by duration:")
df_trips.orderBy(desc("trip_duration_min")).select(
    "trip_id", "trip_duration_min", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime"
).show(5)

print("Shortest trip by duration (excluding 0-minute/negative trips):")
df_trips.filter(col("trip_duration_min") > 0).orderBy("trip_duration_min").select(
    "trip_id", "trip_duration_min", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime"
).show(5)

Longest trip by duration:
+-------+------------------+-------------+--------------------+---------------------+
|trip_id| trip_duration_min|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------+------------------+-------------+--------------------+---------------------+
|  68267| 43648.01666666667|          1.2| 2019-01-01 07:01:20|  2019-01-31 14:29:21|
| 592262|33856.683333333334|          1.1| 2019-01-03 22:24:36|  2019-01-27 10:41:17|
| 875856|           31532.1|          3.9| 2019-01-05 04:21:40|  2019-01-27 01:53:46|
|3715725|            7190.9|          0.0| 2019-01-16 15:31:14|  2019-01-21 15:22:08|
|1715265|1687.0333333333333|         0.78| 2019-01-08 20:18:52|  2019-01-10 00:25:54|
+-------+------------------+-------------+--------------------+---------------------+
only showing top 5 rows
Shortest trip by duration (excluding 0-minute/negative trips):
+-------+--------------------+-------------+--------------------+---------------------+
|trip_id|   trip_duration

#### Busiest day / slowest single day

In [11]:
from pyspark.sql.functions import to_date, count as spark_count

trips_per_day = (
    df_trips.withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date")
    .agg(spark_count("*").alias("trip_count"))
    # January 2019 data occasionally has a handful of mis-dated stray records,
    # so restrict to dates actually within January 2019
    .filter((col("pickup_date") >= "2019-01-01") & (col("pickup_date") <= "2019-01-31"))
    .orderBy(desc("trip_count"))
)

print("Busiest day:")
trips_per_day.show(1)
print("Slowest day:")
trips_per_day.orderBy("trip_count").show(1)

Busiest day:
+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-01-25|    292499|
+-----------+----------+
only showing top 1 row
Slowest day:
+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-01-01|    189432|
+-----------+----------+
only showing top 1 row


#### Busiest / slowest time of day

In [12]:
from pyspark.sql.functions import hour, when

df_trips = df_trips.withColumn("pickup_hour", hour("tpep_pickup_datetime"))

print("Trip counts by hour of day:")
df_trips.groupBy("pickup_hour").agg(spark_count("*").alias("trip_count")) \
    .orderBy("pickup_hour").show(24)

# Bucket hours into broad parts of the day for an easier-to-read summary
df_trips = df_trips.withColumn(
    "day_part",
    when((col("pickup_hour") >= 5) & (col("pickup_hour") < 12), "Morning")
    .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 17), "Afternoon")
    .when((col("pickup_hour") >= 17) & (col("pickup_hour") < 21), "Evening")
    .otherwise("Late Night")
)

print("Trip counts by part of day:")
df_trips.groupBy("day_part").agg(spark_count("*").alias("trip_count")) \
    .orderBy(desc("trip_count")).show()

Trip counts by hour of day:
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|          0|    207842|
|          1|    149254|
|          2|    109421|
|          3|     78086|
|          4|     61424|
|          5|     75533|
|          6|    178598|
|          7|    304858|
|          8|    373742|
|          9|    365935|
|         10|    361390|
|         11|    375441|
|         12|    401173|
|         13|    404153|
|         14|    433139|
|         15|    452691|
|         16|    420843|
|         17|    468479|
|         18|    515390|
|         19|    475186|
|         20|    423156|
|         21|    409901|
|         22|    369041|
|         23|    281941|
+-----------+----------+

Trip counts by part of day:
+----------+----------+
|  day_part|trip_count|
+----------+----------+
| Afternoon|   2111999|
|   Morning|   2035497|
|   Evening|   1882211|
|Late Night|   1666910|
+----------+----------+



#### On average, which day of the week is slowest/busiest

In [13]:
from pyspark.sql.functions import date_format

trips_by_dow = (
    df_trips.withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    .filter((col("pickup_date") >= "2019-01-01") & (col("pickup_date") <= "2019-01-31"))
    .withColumn("day_of_week", date_format("pickup_date", "EEEE"))
    .groupBy("pickup_date", "day_of_week")
    .agg(spark_count("*").alias("trip_count"))
    .groupBy("day_of_week")
    .agg(spark_round(avg("trip_count"), 1).alias("avg_trips_per_day"))
    .orderBy(desc("avg_trips_per_day"))
)

trips_by_dow.show()

+-----------+-----------------+
|day_of_week|avg_trips_per_day|
+-----------+-----------------+
|     Friday|         271787.5|
|   Thursday|         271398.4|
|  Wednesday|         253045.8|
|   Saturday|         252494.8|
|    Tuesday|         241815.2|
|     Monday|         226941.0|
|     Sunday|         214972.5|
+-----------+-----------------+



#### Does trip distance or number of passengers affect tip amount

In [14]:
# corr() computes the Pearson correlation coefficient. Restrict to trips that
# were actually tipped in a sane range to avoid outliers/refunds skewing the result.
tip_trips = df_trips.filter((col("tip_amount") >= 0) & (col("tip_amount") < 200))

distance_corr = tip_trips.stat.corr("trip_distance", "tip_amount")
passenger_corr = tip_trips.stat.corr("passenger_count", "tip_amount")

print(f"Correlation between trip_distance and tip_amount: {distance_corr:.3f}")
print(f"Correlation between passenger_count and tip_amount: {passenger_corr:.3f}")

# A correlation close to 1 (trip_distance) would indicate a strong positive
# relationship, while a correlation close to 0 (passenger_count) would
# indicate little to no relationship.

Correlation between trip_distance and tip_amount: 0.539
Correlation between passenger_count and tip_amount: 0.004


#### What was the highest "extra" charge and which trip

In [15]:
df_trips.orderBy(desc("extra")).select(
    "trip_id", "extra", "fare_amount", "total_amount",
    "tpep_pickup_datetime", "tpep_dropoff_datetime"
).show(5)

+-------+------+-----------+------------+--------------------+---------------------+
|trip_id| extra|fare_amount|total_amount|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------+------+-----------+------------+--------------------+---------------------+
|5323483|535.38|  355676.98|   356214.78| 2019-01-23 08:58:09|  2019-01-23 08:58:09|
|7453230| 23.04|        4.5|       28.34| 2019-01-31 10:06:09|  2019-01-31 10:06:09|
| 311052|  18.5|       52.0|        88.3| 2019-01-02 16:33:28|  2019-01-02 17:17:08|
|2455086|  18.5|       49.0|       96.36| 2019-01-11 16:08:48|  2019-01-11 16:58:11|
| 134549|  18.5|       39.5|        70.8| 2019-01-01 16:09:32|  2019-01-01 16:47:57|
+-------+------+-----------+------------+--------------------+---------------------+
only showing top 5 rows


#### Outliers

In [16]:
# Summary statistics make it easy to spot columns with implausible min/max values
df_trips.select(
    "passenger_count", "trip_distance", "trip_duration_min",
    "fare_amount", "extra", "tip_amount", "total_amount"
).describe().show()

# A few concrete outlier checks:
print("Trips with 0 passengers:", df_trips.filter(col("passenger_count") == 0).count())
print("Trips with negative fare_amount:", df_trips.filter(col("fare_amount") < 0).count())
print("Trips with trip_distance > 100 miles:", df_trips.filter(col("trip_distance") > 100).count())
print("Trips with trip_duration_min > 180 (3+ hours):",
      df_trips.filter(col("trip_duration_min") > 180).count())
print("Trips with trip_duration_min <= 0:", df_trips.filter(col("trip_duration_min") <= 0).count())

+-------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+
|summary|   passenger_count|     trip_distance| trip_duration_min|      fare_amount|             extra|        tip_amount|     total_amount|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+
|  count|           7667945|           7696617|           7696617|          7696617|           7696617|           7696617|          7696617|
|   mean|1.5670317144945614|2.8301461681153532|16.551081570422276|12.52967677747685|0.3374054146126797|1.8208300763883147|15.81065134371489|
| stddev|1.2244198591042095| 3.774548394256295| 81.67539611217202|261.5897471783846|0.5313564053935059|2.4994631914320986|261.8117056584905|
|    min|               0.0|               0.0|          -84280.5|           -362.0|             -60.0|             -63.5|           -362.8|
|    max|    

**Outlier reasoning:** Several columns contain values that are not physically
plausible for a taxi trip and should be treated as outliers/data-quality
issues rather than genuine records: trips with `passenger_count == 0`
(no rider on record), a negative `fare_amount` or `total_amount` (likely
refunds/corrections rather than real trips), a `trip_distance` of several
hundred miles (far beyond anything possible within NYC and its surrounding
area), and a `trip_duration_min` that is zero/negative (the dropoff
timestamp is before or equal to the pickup timestamp, an impossible trip)
or extremely large (multi-hour "trips" that are more likely a driver
forgetting to close out a trip in the meter). In a production analysis
these rows would typically be filtered out or flagged before computing
aggregate statistics, since a handful of them can meaningfully skew
averages such as `avg(trip_distance)` or `avg(trip_duration_min)`.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

#### Load the taxi zone lookup table

In [17]:
# set dl url for the taxi zone lookup table
zone_lookup_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# get the data
response = requests.get(zone_lookup_url)

# check that response was good and save the data
zone_lookup_file = "taxi_zone_lookup.csv"
if response.status_code == 200:
    with open(zone_lookup_file, "wb") as f:
        f.write(response.content)

In [18]:
# create the zone lookup dataframe. This file is small and simple enough that
# schema inference over the whole file is fine (no samplingRatio needed).
df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_lookup_file)

df_zones.show(5)
df_zones.printSchema()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



#### Which borough had the most pickups? Dropoffs?

In [19]:
# Join on PULocationID / DOLocationID to bring in the Borough for each trip
df_trips_zoned = (
    df_trips
    .join(df_zones.select(col("LocationID").alias("PULocationID"), col("Borough").alias("PU_Borough")),
          on="PULocationID", how="left")
    .join(df_zones.select(col("LocationID").alias("DOLocationID"), col("Borough").alias("DO_Borough")),
          on="DOLocationID", how="left")
)

print("Pickups by borough:")
df_trips_zoned.groupBy("PU_Borough").agg(spark_count("*").alias("pickup_count")) \
    .orderBy(desc("pickup_count")).show()

print("Dropoffs by borough:")
df_trips_zoned.groupBy("DO_Borough").agg(spark_count("*").alias("dropoff_count")) \
    .orderBy(desc("dropoff_count")).show()

Pickups by borough:
+-------------+------------+
|   PU_Borough|pickup_count|
+-------------+------------+
|    Manhattan|     6950965|
|       Queens|      471173|
|      Unknown|      159815|
|     Brooklyn|       91905|
|        Bronx|       18062|
|          N/A|        3890|
|          EWR|         446|
|Staten Island|         361|
+-------------+------------+

Dropoffs by borough:
+-------------+-------------+
|   DO_Borough|dropoff_count|
+-------------+-------------+
|    Manhattan|      6817355|
|       Queens|       340972|
|     Brooklyn|       301105|
|      Unknown|       149097|
|        Bronx|        58085|
|          N/A|        16904|
|          EWR|        10914|
|Staten Island|         2185|
+-------------+-------------+



#### Busy / slow times by borough

In [20]:
df_trips_zoned.groupBy("PU_Borough", "day_part") \
    .agg(spark_count("*").alias("trip_count")) \
    .orderBy("PU_Borough", desc("trip_count")) \
    .show(40, truncate=False)

+-------------+----------+----------+
|PU_Borough   |day_part  |trip_count|
+-------------+----------+----------+
|Bronx        |Morning   |8407      |
|Bronx        |Afternoon |4541      |
|Bronx        |Late Night|2602      |
|Bronx        |Evening   |2512      |
|Brooklyn     |Morning   |31544     |
|Brooklyn     |Late Night|26171     |
|Brooklyn     |Afternoon |18873     |
|Brooklyn     |Evening   |15317     |
|EWR          |Afternoon |206       |
|EWR          |Morning   |120       |
|EWR          |Evening   |91        |
|EWR          |Late Night|29        |
|Manhattan    |Afternoon |1912287   |
|Manhattan    |Morning   |1836840   |
|Manhattan    |Evening   |1713230   |
|Manhattan    |Late Night|1488608   |
|N/A          |Late Night|1256      |
|N/A          |Morning   |1008      |
|N/A          |Afternoon |870       |
|N/A          |Evening   |756       |
|Queens       |Afternoon |130169    |
|Queens       |Morning   |115882    |
|Queens       |Late Night|113936    |
|Queens     

#### Busiest days of the week by borough

In [21]:
trips_by_dow_borough = (
    df_trips_zoned
    .withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    .filter((col("pickup_date") >= "2019-01-01") & (col("pickup_date") <= "2019-01-31"))
    .withColumn("day_of_week", date_format("pickup_date", "EEEE"))
    .groupBy("PU_Borough", "day_of_week")
    .agg(spark_count("*").alias("trip_count"))
    .orderBy("PU_Borough", desc("trip_count"))
)

trips_by_dow_borough.show(50, truncate=False)

+-------------+-----------+----------+
|PU_Borough   |day_of_week|trip_count|
+-------------+-----------+----------+
|Bronx        |Thursday   |3121      |
|Bronx        |Tuesday    |3059      |
|Bronx        |Wednesday  |2998      |
|Bronx        |Friday     |2666      |
|Bronx        |Monday     |2172      |
|Bronx        |Sunday     |2112      |
|Bronx        |Saturday   |1928      |
|Brooklyn     |Tuesday    |15779     |
|Brooklyn     |Thursday   |15714     |
|Brooklyn     |Wednesday  |15101     |
|Brooklyn     |Friday     |13092     |
|Brooklyn     |Saturday   |11604     |
|Brooklyn     |Sunday     |11099     |
|Brooklyn     |Monday     |9507      |
|EWR          |Wednesday  |83        |
|EWR          |Tuesday    |77        |
|EWR          |Friday     |74        |
|EWR          |Sunday     |68        |
|EWR          |Thursday   |58        |
|EWR          |Saturday   |55        |
|EWR          |Monday     |31        |
|Manhattan    |Thursday   |1229516   |
|Manhattan    |Wednesday 

#### Average trip distance by borough

In [22]:
df_trips_zoned.groupBy("PU_Borough") \
    .agg(spark_round(avg("trip_distance"), 2).alias("avg_trip_distance")) \
    .orderBy(desc("avg_trip_distance")).show()

+-------------+-----------------+
|   PU_Borough|avg_trip_distance|
+-------------+-----------------+
|Staten Island|             12.5|
|       Queens|            11.28|
|        Bronx|             7.23|
|     Brooklyn|             4.79|
|          N/A|             3.19|
|          EWR|             2.64|
|      Unknown|             2.42|
|    Manhattan|             2.23|
+-------------+-----------------+



#### Average trip fare by borough

In [23]:
df_trips_zoned.groupBy("PU_Borough") \
    .agg(spark_round(avg("fare_amount"), 2).alias("avg_fare_amount")) \
    .orderBy(desc("avg_fare_amount")).show()

+-------------+---------------+
|   PU_Borough|avg_fare_amount|
+-------------+---------------+
|          EWR|          76.24|
|          N/A|          59.57|
|Staten Island|          45.29|
|       Queens|          35.14|
|        Bronx|          26.27|
|     Brooklyn|          18.65|
|      Unknown|          14.94|
|    Manhattan|          10.79|
+-------------+---------------+



#### Highest / lowest fare amounts for a trip, and the associated borough

In [24]:
print("Highest fare trip:")
df_trips_zoned.orderBy(desc("fare_amount")).select(
    "trip_id", "fare_amount", "PU_Borough", "DO_Borough", "trip_distance"
).show(5)

print("Lowest fare trip (excluding negative/refund fares):")
df_trips_zoned.filter(col("fare_amount") >= 0).orderBy("fare_amount").select(
    "trip_id", "fare_amount", "PU_Borough", "DO_Borough", "trip_distance"
).show(5)

Highest fare trip:
+-------+-----------+----------+----------+-------------+
|trip_id|fare_amount|PU_Borough|DO_Borough|trip_distance|
+-------+-----------+----------+----------+-------------+
|2499655|  623259.86| Manhattan| Manhattan|          2.4|
|5323483|  355676.98| Manhattan|   Unknown|          0.0|
|2159971|    36090.3|   Unknown|   Unknown|          0.0|
|1892781|   34674.65|   Unknown|   Unknown|          0.0|
|1649451|   33023.53|   Unknown|   Unknown|          0.0|
+-------+-----------+----------+----------+-------------+
only showing top 5 rows
Lowest fare trip (excluding negative/refund fares):
+-------+-----------+----------+----------+-------------+
|trip_id|fare_amount|PU_Borough|DO_Borough|trip_distance|
+-------+-----------+----------+----------+-------------+
|  13256|        0.0|    Queens|    Queens|          0.0|
|  16634|        0.0| Manhattan| Manhattan|          0.0|
|  13257|        0.0|    Queens|    Queens|          0.0|
|  12508|        0.0|       N/A|   

#### Load January 2025 data and compare

In [25]:
# set dl url for January 2025 trip data
download_url_2025 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"

response = requests.get(download_url_2025)

jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"
if response.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

df_trips_2025 = spark.read.parquet(jan_2025_trip_data)
df_trips_2025.cache()
df_trips_2025.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [26]:
# Join the 2025 data to boroughs the same way as 2019 so the comparison is
# apples-to-apples
df_trips_2025_zoned = (
    df_trips_2025
    .join(df_zones.select(col("LocationID").alias("PULocationID"), col("Borough").alias("PU_Borough")),
          on="PULocationID", how="left")
    .join(df_zones.select(col("LocationID").alias("DOLocationID"), col("Borough").alias("DO_Borough")),
          on="DOLocationID", how="left")
)

metrics_2019 = df_trips_zoned.agg(
    spark_round(avg("trip_distance"), 2).alias("avg_trip_distance"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare_amount"),
    spark_round(avg("passenger_count"), 2).alias("avg_passenger_count"),
    spark_round(avg("tip_amount"), 2).alias("avg_tip_amount"),
).withColumn("year", col("avg_trip_distance") * 0 + 2019)

metrics_2025 = df_trips_2025_zoned.agg(
    spark_round(avg("trip_distance"), 2).alias("avg_trip_distance"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare_amount"),
    spark_round(avg("passenger_count"), 2).alias("avg_passenger_count"),
    spark_round(avg("tip_amount"), 2).alias("avg_tip_amount"),
).withColumn("year", col("avg_trip_distance") * 0 + 2025)

metrics_2019.union(metrics_2025).select(
    "year", "avg_trip_distance", "avg_fare_amount", "avg_passenger_count", "avg_tip_amount"
).show()

+------+-----------------+---------------+-------------------+--------------+
|  year|avg_trip_distance|avg_fare_amount|avg_passenger_count|avg_tip_amount|
+------+-----------------+---------------+-------------------+--------------+
|2019.0|             2.83|          12.53|               1.57|          1.82|
|2025.0|             5.86|          17.08|                1.3|          2.96|
+------+-----------------+---------------+-------------------+--------------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

For this section the main lab was completed with PySpark, so the three
questions below are re-answered using pure Spark SQL instead. Temp views are
registered first so that plain SQL can query the dataframes.

In [27]:
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

#### SQL: What is the average passenger count

In [28]:
spark.sql("""
    SELECT ROUND(AVG(passenger_count), 2) AS avg_passenger_count
    FROM trips
""").show()

+-------------------+
|avg_passenger_count|
+-------------------+
|               1.57|
+-------------------+



#### SQL: Which trip has the highest passenger count

In [29]:
spark.sql("""
    SELECT trip_id, passenger_count, tpep_pickup_datetime, tpep_dropoff_datetime,
           trip_distance, total_amount
    FROM trips
    ORDER BY passenger_count DESC
    LIMIT 5
""").show()

+-------+---------------+--------------------+---------------------+-------------+------------+
|trip_id|passenger_count|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|total_amount|
+-------+---------------+--------------------+---------------------+-------------+------------+
|2012098|            9.0| 2019-01-10 00:43:10|  2019-01-10 00:43:14|          0.0|        11.3|
|4997790|            9.0| 2019-01-21 19:20:28|  2019-01-21 19:20:31|          0.0|         9.8|
|2883995|            9.0| 2019-01-13 04:13:24|  2019-01-13 04:14:34|          0.0|       12.25|
| 949956|            9.0| 2019-01-05 13:12:29|  2019-01-05 13:12:32|          0.0|        12.6|
|4534707|            9.0| 2019-01-19 16:45:25|  2019-01-19 16:45:27|          0.0|      110.76|
+-------+---------------+--------------------+---------------------+-------------+------------+



#### SQL: Which borough had the most pickups (join)

In [30]:
spark.sql("""
    SELECT z.Borough AS pu_borough, COUNT(*) AS pickup_count
    FROM trips t
    JOIN zones z
      ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY pickup_count DESC
""").show()

+-------------+------------+
|   pu_borough|pickup_count|
+-------------+------------+
|    Manhattan|     6950965|
|       Queens|      471173|
|      Unknown|      159815|
|     Brooklyn|       91905|
|        Bronx|       18062|
|          N/A|        3890|
|          EWR|         446|
|Staten Island|         361|
+-------------+------------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing

#### Build out the rest of 2019 and find the busiest season

In [31]:
# Download every remaining month of 2019 the same way January was downloaded,
# then union everything together and tag each row with its season.
month_urls = {
    m: f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-{m:02d}.parquet"
    for m in range(1, 13)
}

monthly_dfs = []
for month, url in month_urls.items():
    local_file = f"yellow_tripdata_2019-{month:02d}.parquet"
    resp = requests.get(url)
    if resp.status_code == 200:
        with open(local_file, "wb") as f:
            f.write(resp.content)
        monthly_dfs.append(spark.read.parquet(local_file))
    else:
        print(f"Skipped month {month:02d}, HTTP status {resp.status_code}")

df_trips_2019_full = monthly_dfs[0]
for m_df in monthly_dfs[1:]:
    df_trips_2019_full = df_trips_2019_full.unionByName(m_df, allowMissingColumns=True)

df_trips_2019_full.cache()
print(f"Total 2019 trips loaded: {df_trips_2019_full.count():,}")

Total 2019 trips loaded: 84,598,444


In [32]:
# Meteorological seasons (Northern Hemisphere):
# Winter: Dec, Jan, Feb | Spring: Mar, Apr, May | Summer: Jun, Jul, Aug | Fall: Sep, Oct, Nov
from pyspark.sql.functions import month as spark_month

df_trips_2019_seasoned = df_trips_2019_full.withColumn(
    "season",
    when(spark_month("tpep_pickup_datetime").isin(12, 1, 2), "Winter")
    .when(spark_month("tpep_pickup_datetime").isin(3, 4, 5), "Spring")
    .when(spark_month("tpep_pickup_datetime").isin(6, 7, 8), "Summer")
    .otherwise("Fall")
)

df_trips_2019_seasoned.groupBy("season") \
    .agg(spark_count("*").alias("trip_count")) \
    .orderBy(desc("trip_count")).show()

+------+----------+
|season|trip_count|
+------+----------+
|Spring|  22941027|
|Winter|  21643025|
|  Fall|  20659565|
|Summer|  19354827|
+------+----------+



#### Visualize 3 questions using Spark's native (v4+) plotting support

In [33]:
# Spark 4 dataframes support .plot directly (backed by plotly). If running on
# an older Spark version, fall back to .toPandas().plot(...) with matplotlib instead.

# 1. Trip counts by hour of day
hourly_counts = df_trips.groupBy("pickup_hour").agg(spark_count("*").alias("trip_count")) \
    .orderBy("pickup_hour")
hourly_counts.plot.bar(x="pickup_hour", y="trip_count")

In [34]:
# 2. Average fare amount by borough
fare_by_borough = df_trips_zoned.groupBy("PU_Borough") \
    .agg(spark_round(avg("fare_amount"), 2).alias("avg_fare_amount")) \
    .orderBy(desc("avg_fare_amount"))
fare_by_borough.plot.bar(x="PU_Borough", y="avg_fare_amount")

In [35]:
# 3. Trip distance vs. tip amount (sampled, since plotting the full dataset
# as a scatter plot would be far too dense to read)
scatter_sample = tip_trips.filter(col("trip_distance") < 30).sample(fraction=0.001, seed=42)
scatter_sample.plot.scatter(x="trip_distance", y="tip_amount")

#### Explore a dataset of your choosing

In [ ]:
# Open cell for further, self-directed exploration - e.g. green/FHV trip
# data, weather data joined against pickup dates, or a completely different
# dataset of interest.
